# 06 — The Complete Pipeline

This notebook uses **only the public API**: `MapEncryption`, `SchemeParams`, `SCHEME_VERSION`, and `_R_EARTH` (for the `metres_to_deg` helper). No internal functions are imported.

We encode 500 synthetic GPS points near Times Square, render display positions, decode back to exact coordinates, and verify round-trip fidelity — all with interactive Plotly maps.

In [2]:
import secrets
import math
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from map_encryption import MapEncryption, SchemeParams, SCHEME_VERSION, _R_EARTH

def metres_to_deg(spread_m, at_lat):
    lat_deg = spread_m / 111_320
    lon_deg = spread_m / (111_320 * math.cos(math.radians(at_lat)))
    return lat_deg, lon_deg

# in production: load from secrets manager
MASTER_KEY = secrets.token_bytes(32)

CENTER_LAT, CENTER_LON = 40.758, -73.985  # Times Square
enc = MapEncryption(MASTER_KEY, SchemeParams())

p = enc.params
print(f'Scheme version:    {SCHEME_VERSION}')
print(f'bin_size_m:        {p.bin_size_m} m')
print(f'jitter_max_frac:   {p.jitter_max_frac}')
print(f'prp_rounds:        {p.prp_rounds}')

Scheme version:    1
bin_size_m:        250 m
jitter_max_frac:   0.25
prp_rounds:        10


## Record Schema

| Field | Type | Size | Is it secret? | Purpose |
|-------|------|------|---------------|---------|
| `qxp` | int | 4–8 bytes | No — but opaque | Shuffled tile x-index; reveals nothing without PRP key |
| `qyp` | int | 4–8 bytes | No — but opaque | Shuffled tile y-index; reveals nothing without PRP key |
| `nonce` | bytes | 12 bytes | No — public | Unique per encode call; seeds AEAD and display jitter |
| `ct_resid` | bytes | 32 bytes | **Yes** | AEAD ciphertext of (rx, ry); holds full GPS precision |
| `tweak` | bytes | variable | No — public | Record-ID context; binds ciphertext to this record |
| `version` | int | 1 byte | No | Scheme version for forward-compatible migration |

In [3]:
rng = np.random.default_rng(42)
n = 500
spread_m = 3000
dlat_s, dlon_s = metres_to_deg(spread_m, CENTER_LAT)

lats = CENTER_LAT + rng.uniform(-dlat_s, dlat_s, n)
lons = CENTER_LON + rng.uniform(-dlon_s, dlon_s, n)

records = [
    enc.encode(lats[i], lons[i],
               tweak=MapEncryption.make_tweak(record_id=i, extra=b'nb06-demo'))
    for i in range(n)
]

render_pos = [enc.render_coordinates(r) for r in records]
print(f'Encoded {len(records)} records.')
print(f'Sample record keys: {list(records[0].keys())}')

Encoded 500 records.
Sample record keys: ['qxp', 'qyp', 'nonce', 'ct_resid', 'tweak', 'version']


In [4]:
decoded = [enc.decode(r) for r in records]
assert all(d is not None for d in decoded), 'decode returned None unexpectedly'

max_error = max(
    max(abs(decoded[i][0] - lats[i]), abs(decoded[i][1] - lons[i]))
    for i in range(n)
)
assert max_error < 1e-9, f'Round-trip error {max_error} exceeds tolerance'
print(f'Max round-trip error: {max_error:.2e} degrees — exact lossless encoding confirmed')

Max round-trip error: 3.55e-14 degrees — exact lossless encoding confirmed


In [5]:
import pandas as pd

rows = []
for i in range(n):
    rows.append({'lat': lats[i], 'lon': lons[i], 'label': 'Original'})
    rows.append({'lat': render_pos[i][0], 'lon': render_pos[i][1], 'label': 'Display (render_coordinates)'})
    rows.append({'lat': decoded[i][0], 'lon': decoded[i][1], 'label': 'Decoded'})

df_three = pd.DataFrame(rows)
fig = px.scatter_mapbox(
    df_three, lat='lat', lon='lon', color='label',
    mapbox_style='open-street-map',
    zoom=12,
    center={'lat': CENTER_LAT, 'lon': CENTER_LON},
    title='Three views of 500 encrypted points',
    height=500,
    color_discrete_map={
        'Original': 'steelblue',
        'Display (render_coordinates)': 'orange',
        'Decoded': 'green'
    },
    opacity=0.7
)
fig.show()

/var/folders/gl/ppzrtfy95t7ghxlyd9tf3vk00000gn/T/ipykernel_73847/1765662777.py:10: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


In [5]:
# Sample 50 records for line traces (reduce visual clutter)
sample_idx = list(range(0, n, n // 50))[:50]

fig2 = go.Figure()

# Lines from render position to decoded position (50 sampled)
for i in sample_idx:
    fig2.add_trace(go.Scattermapbox(
        lat=[render_pos[i][0], decoded[i][0]],
        lon=[render_pos[i][1], decoded[i][1]],
        mode='lines',
        line=dict(color='orange', width=1),
        showlegend=False
    ))

# All 500 decode positions as scatter
fig2.add_trace(go.Scattermapbox(
    lat=[d[0] for d in decoded],
    lon=[d[1] for d in decoded],
    mode='markers',
    marker=dict(size=5, color='steelblue'),
    name='decoded (all 500)'
))

fig2.update_layout(
    mapbox_style='open-street-map',
    mapbox_zoom=12,
    mapbox_center={'lat': CENTER_LAT, 'lon': CENTER_LON},
    title='Jitter displacement: render position -> decoded position (50 sampled)',
    height=500
)
fig2.show()

/var/folders/gl/ppzrtfy95t7ghxlyd9tf3vk00000gn/T/ipykernel_2621/3422890986.py:8: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig2.add_trace(go.Scattermapbox(
/var/folders/gl/ppzrtfy95t7ghxlyd9tf3vk00000gn/T/ipykernel_2621/3422890986.py:17: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig2.add_trace(go.Scattermapbox(


In [6]:
errors = [
    max(abs(decoded[i][0] - lats[i]), abs(decoded[i][1] - lons[i]))
    for i in range(n)
]

fig3 = px.histogram(
    x=errors,
    nbins=30,
    labels={'x': 'Max coordinate error (degrees)', 'y': 'Count'},
    title='Round-trip error distribution across 500 records',
    template='plotly_white',
    height=400
)
fig3.show()

print(f'All errors at or below: {max(errors):.2e} degrees')

All errors at or below: 3.55e-14 degrees


In [6]:
import struct as _struct

r0 = records[0]
wrong_enc = MapEncryption(secrets.token_bytes(32), SchemeParams())

scenarios = [
    ('Flip ct byte',
     lambda: enc.decode({**r0, 'ct_resid': bytes([r0['ct_resid'][0]^1]) + r0['ct_resid'][1:]})),
    ('Flip nonce byte',
     lambda: enc.decode({**r0, 'nonce': bytes([r0['nonce'][0]^1]) + r0['nonce'][1:]})),
    ('Shift qxp',
     lambda: enc.decode({**r0, 'qxp': r0['qxp'] + 1})),
    ('Wrong tweak',
     lambda: enc.decode({**r0, 'tweak': b'wrong-tweak'})),
    ('Wrong version',
     lambda: enc.decode({**r0, 'version': 999})),
    ('Wrong master key',
     lambda: wrong_enc.decode(r0)),
]

print(f'{"Scenario":<22} {"Result"}')
print('-' * 38)
all_none = True
for name, fn in scenarios:
    result = fn()
    status = 'PASS (None)' if result is None else 'FAIL'
    if result is not None:
        all_none = False
    print(f'  {name:<20} {status}')

assert all_none, 'At least one failure mode was not rejected'
print('\nAll 6 failure mode scenarios correctly returned None.')

Scenario               Result
--------------------------------------
  Flip ct byte         PASS (None)
  Flip nonce byte      PASS (None)
  Shift qxp            PASS (None)
  Wrong tweak          PASS (None)
  Wrong version        PASS (None)
  Wrong master key     PASS (None)

All 6 failure mode scenarios correctly returned None.


## Parameter Tuning

All participants in a deployment must use **identical** `SchemeParams`. A mismatch in any parameter will cause decode to fail silently (wrong tile recovery leads to wrong AD, so AEAD decryption returns None).

| Parameter | Default | Effect of increasing | Effect of decreasing |
|-----------|---------|----------------------|----------------------|
| `bin_size_m` | 250 m | Coarser spatial privacy; fewer total tiles | Finer grid; more tiles; less spatial masking |
| `jitter_max_frac` | 0.25 | Display pins spread further from tile centre | Display pins cluster near tile centre |
| `prp_rounds` | 10 | More cryptographic security for PRP; slower | Faster; below 4 rounds, security proofs weaken |